In [ ]:
    ############    #############   Middleware and Background Tasks   #############   ##############   

 =>  Middleware wraps EVERY request/response -- use it for cross-cutting concerns (auth
       checks, request timing, correlation IDs) that don't belong in any single route.

 =>  A background task runs AFTER the response has already been sent to the client -- use
       it for work the client shouldn't have to wait on (sending an email, logging
       analytics) but that doesn't need a full task queue (see Phase 1's Distributed
       Systems topic for when you DO need a real queue).


<img src="images/request-pipeline.png" alt="Request pipeline: middleware chain then route handler then response, with a background task after">

In [ ]:
import time
from fastapi import FastAPI, Request, BackgroundTasks
from fastapi.testclient import TestClient

app = FastAPI()
events: list[str] = []

@app.middleware("http")
async def add_timing_header(request: Request, call_next):
    start = time.perf_counter()
    events.append(f"middleware: before {request.url.path}")
    response = await call_next(request)
    duration_ms = (time.perf_counter() - start) * 1000
    response.headers["X-Process-Time-Ms"] = f"{duration_ms:.2f}"
    events.append(f"middleware: after {request.url.path}")
    return response

def log_analytics(path: str):
    events.append(f"background task: logged visit to {path}")

@app.get("/items/{item_id}")
def get_item(item_id: int, background_tasks: BackgroundTasks):
    background_tasks.add_task(log_analytics, f"/items/{item_id}")
    events.append(f"route handler: fetching item {item_id}")
    return {"item_id": item_id}

client = TestClient(app)
response = client.get("/items/42")
print("status:", response.status_code)
print("X-Process-Time-Ms header present:", "X-Process-Time-Ms" in response.headers)
print("execution order:")
for e in events:
    print(" ", e)


In [ ]:
 =>  The execution order shows the middleware running BEFORE the route handler, the
       route handler running, the middleware finishing AFTER (adding the timing header),
       and the background task logging AFTER that -- matching the diagram above exactly.

 =>  TestClient runs background tasks synchronously before returning, so you can assert on
       their effects in tests -- in a real deployment they run on the event loop after the
       response bytes are already on the wire.


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Add a second middleware that rejects requests missing an 'Authorization' header
           with a 401, and confirm it runs before the route handler.

 =>  [ ] Replace the background task with a real one (e.g. writing a line to a log file)
           and confirm it doesn't add latency to the response itself.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Doing slow/blocking work inside middleware -- it runs on EVERY request, so any latency
       added here is paid by every single request, not just the ones that need it.

 =>  Using BackgroundTasks for anything that must survive a server restart or that needs
       retries -- it's in-process and best-effort; use a real task queue (Celery, arq, an
       actual message broker) for anything that must not be silently lost.
